# 🕷️ Web Scraping with Python
## BeautifulSoup · requests · Selenium — explained in detail

> **Notebook goal:** Learn exactly how the web works over HTTP, how to download pages
> with `requests`, parse and extract data with `BeautifulSoup`, and when (and how) to
> render JavaScript-heavy pages with `Selenium`.

This notebook runs **fully offline**: every demo uses a small in-memory HTML string, so it
works anywhere. Real network steps are shown as commented examples you can uncomment and run
when you have an internet connection.

**What you will be able to do by the end:**
- Fetch a webpage's HTML with `requests`
- Understand URLs, requests, responses and status codes
- Read and write HTML & CSS selectors
- Parse HTML with BeautifulSoup and extract text / attributes
- Turn scraped data into a pandas `DataFrame`
- Handle errors, retries, and pagination like a professional
- Use `Selenium` for JavaScript-rendered pages
- Respect the law and the site's terms of service



## Table of Contents
1. **Why scrape?** — when and why web scraping matters
2. **How the web works** — URLs, HTTP requests/responses, status codes
3. **The `requests` library** — download HTML, headers, sessions
4. **HTML & the DOM** — tags, attributes, nesting
5. **CSS selectors** — how to target elements
6. **BeautifulSoup fundamentals** — parse, find, select
7. **Extracting data** — text, attributes, loops
8. **From HTML to a DataFrame** — the full mini-project
9. **Pagination** — scraping multiple pages
10. **Robust scraping** — errors, retries, rate limiting
11. **Selenium** — scraping dynamic / JavaScript sites
12. **Ethics & legality** — robots.txt, terms, rate limits
13. **Glossary & cheatsheet**
14. **Practice problems**



## 1 · Why Web Scraping?

**Web scraping** = automatically extracting data from websites using a program
(instead of copying and pasting by hand).

### When you need it
| Situation | Solution |
|-----------|----------|
| The data exists only on a website (no API/file) | Scrape the HTML |
| You need data repeated every hour/day | Automate the download |
| You need thousands of rows | Program beats manual copying |
| Site has an API | **Prefer the API** (faster, legal, stable) |

### Scrape vs API — decision rule
> **Always check for an official API first.** Scraping is the last resort when no API
> or downloadable dataset exists. APIs are faster, more reliable, and don't break the rules.



## 2 · How the Web Works (HTTP)

When you scrape, you are simply **acting like a browser** — asking a server for pages.

### The URL
```
https://  www.example.com  /products  ?category=phone  &page=2
   |            |              |             |            |
 scheme      host          path        query params     (more params)
```
- **Scheme** — `http` / `https` (secure)
- **Host** — the website's server address
- **Path** — which resource on the server
- **Query string** — extra options after `?`, separated by `&`

### The request → response cycle
```
Your program  ---(HTTP request)-->  Web server
Your program  <--(HTTP response)--  Web server (HTML / JSON / ...)
```
A request has:
- a **method** (`GET` = fetch, `POST` = send data)
- a **URL**
- **headers** (identity, e.g. `User-Agent`)
- optionally **parameters** in the URL or body

A response has:
- a **status code** (200 ok, 404 not found, 403 forbidden...)
- **headers** (content-type, cookies...)
- a **body** (the actual HTML / JSON / file)

### Common status codes
| Code | Meaning | Handle by |
|------|---------|-----------|
| 200 | OK | parse the body |
| 301 / 302 | redirect | let `requests` follow it |
| 403 | forbidden / blocked | add headers, slow down, use proxy |
| 404 | not found | skip / log it |
| 429 | too many requests | **sleep and retry** |
| 500 / 503 | server error | retry after a pause |



In [ ]:
import requests

# What a real request + response looks like (explore on your own with internet):
# r = requests.get("https://httpbin.org/get")
# print(r.status_code)      # 200
# print(r.headers.get("Content-Type"))
# print(r.text[:200])       # first 200 chars of the body
print("requests version:", requests.__version__)


## 3 · The `requests` Library

`requests` is the de-facto Python library for HTTP. It is **not** an HTML parser —
its job is to *download* the page. BeautifulSoup (section 6) parses it.

### Basic pattern
```python
import requests
r = requests.get("https://example.com")
r.text          # the HTML as a string
r.status_code   # e.g. 200
r.headers       # response headers (dict)
r.url           # final URL (after redirects)
r.raise_for_status()  # raise an error for 4xx/5xx
```

### Be a polite browser
Many servers reject requests that have no `User-Agent` (they look like bots):
```python
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
r = requests.get(url, headers=headers, timeout=10)
```
- **Always set `timeout`** — a hanging request is a hanging script.
- **Use a `Session`** for multiple requests to reuse cookies and `Connection` pooling.



In [ ]:
import requests

# A polite downloader function. Call it only when online.
def polite_get(url, headers=None, timeout=10):
    h = headers or {"User-Agent": "Mozilla/5.0 (educational scraping notebook)"}
    r = requests.get(url, headers=h, timeout=timeout)
    r.raise_for_status()          # raises for 4xx / 5xx
    return r

# Example (offline-safe — try it when you have internet):
# html_page = polite_get("https://books.toscrape.com/")
# print(html_page.status_code, len(html_page.text), "bytes")
print("polite_get() defined — uses a browser-like User-Agent and a timeout.")


In [ ]:
import requests

# Sessions keep cookies between requests (needed for logins / carts).
# session = requests.Session()
# session.get("https://example.com/login")
# session.post("https://example.com/login", data={"user": "me", "pass": "x"})
# profile = session.get("https://example.com/account")
# print(profile.text[:200])

# Headers you can inspect / send
headers = {
    "User-Agent": "Mozilla/5.0 ...",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html",
}
print("A realistic headers dict:", headers)


## 4 · HTML & the DOM

**HTML** (HyperText Markup Language) describes webpage *structure* with **tags**.

```html
<html>
  <body>
    <h1 class="title">Products</h1>
    <div class="card">
      <h2>Phone X</h2>
      <p class="price">15000</p>
      <a href="/p/1">See details</a>
    </div>
  </body>
</html>
```

### Anatomy of a tag
```
<  p  class="price"   >  15000   </  p  >
 |   |        |        |          |
open  tagname  attr     content   close
tag             value             tag
```
- **Element** = open tag + content + close tag
- **Attributes** = metadata inside the open tag (`class`, `id`, `href`, `src`...)
- **Nesting** = tags inside tags → this forms the **DOM tree**
- The **DOM** (Document Object Model) is how browsers (and BeautifulSoup) view the page as a tree of elements.

### Tags you will meet most
| Tag | Purpose |
|-----|---------|
| `<div>` / `<span>` | generic containers (used for layout/cards) |
| `<h1>`–`<h6>` | headings |
| `<p>` | paragraph text |
| `<a href>` | hyperlink |
| `<img src>` | image |
| `<table>` `<tr>` `<td>` | tables |
| `<ul>` `<li>` | lists |



## 5 · CSS Selectors

To *find* data in HTML, you describe **where** it is. CSS selectors are the most
common language for that.

| Selector | Matches | Example |
|----------|---------|---------|
| `tagname` | all tags of that name | `p`, `h2`, `div` |
| `.class` | elements with that class | `.price`, `.card` |
| `#id` | the element with that id | `#main` |
| `tag.class` | tag with both | `p.price` |
| `parent child` | child inside parent | `div.card h2` |
| `parent > child` | **direct** child | `ul > li` |
| `tag, tag` | either | `h1, h2` |
| `[attr]` | has the attribute | `[href]` |
| `[attr=value]` | attribute equals value | `a[href="/p/1"]` |

### Rule of thumb
> The `class` attribute is the web developer's way of labeling elements for styling —
> which makes it the **most useful** hook for scraping too. Look for repeated,
> descriptive classes: `.card`, `.price`, `.title`, `.item`.



In [ ]:
# Practice reading selectors — predict what each matches (answers in the markdown below)
selectors = {
    "div":                "all <div> tags",
    ".price":             "any element with class='price'",
    "#main":              "the element with id='main'",
    "div.card h2":        "an h2 inside a div.card",
    "ul > li":            "a DIRECT li child of a ul",
    "p, span":            "all p OR all span",
    "a[href]":            "anchors that have an href",
    "table tr td":        "table cells inside rows inside a table",
}
for sel, desc in selectors.items():
    print(f"{sel:12} -> {desc}")


## 6 · BeautifulSoup Fundamentals

**BeautifulSoup4** (`bs4`) parses messy HTML into a navigable tree and lets you
`find` things with tags, classes, attributes, or CSS selectors.

### Install
```bash
pip install beautifulsoup4
```
(Optionally `pip install lxml` for a faster parser — in this notebook we use the
built-in `html.parser` so it runs everywhere.)

### Import & parse
```python
from bs4 import BeautifulSoup
soup = BeautifulSoup(html_string, "html.parser")
```

### The three things you will do 90% of the time
1. `soup.find("tag")` → **first** match
2. `soup.find_all("tag")` → **list** of all matches
3. `soup.select("css")` → matches by CSS selector

Let's demonstrate each on a small page.



In [ ]:
from bs4 import BeautifulSoup

html = '''<html><body>
  <h1>Online Store</h1>
  <p class="price">100</p>
  <p class="price">250</p>
  <a href="/p/1">Phone</a>
  <a href="/p/2">Mouse</a>
</body></html>'''

soup = BeautifulSoup(html, "html.parser")

print("soup.title :", soup.title)      # first <title> (None here)
print("1st <p>    :", soup.find("p"))
print("all <p>    :", soup.find_all("p"))
print("all <a>    :", soup.find_all("a"))


In [ ]:
from bs4 import BeautifulSoup

html = '''<div id="main">
  <div class="card"><h2>Phone X</h2><p class="price">15000</p></div>
  <div class="card"><h2>Phone Y</h2><p class="price">20000</p></div>
  <div class="card"><h2>Phone Z</h2><p class="price">25000</p></div>
</div>'''

soup = BeautifulSoup(html, "html.parser")

cards = soup.select("div.card")               # every card container
print("number of cards:", len(cards))

first = soup.select_one("div.card h2")        # first h2 inside a card
print("first card h2  :", first.text)

prices = soup.select("p.price")               # all price tags
print("prices         :", [p.text for p in prices])


## 7 · Extracting Data

### Text vs attributes
```python
element.text            # the visible text inside the element
element["href"]         # the value of the 'href' attribute
element.get("class")    # safely get an attribute (None if missing)
element.name            # the tag name
```

### `text` vs `get_text()`
Both return the inner text; `get_text(separator=" ", strip=True)` cleans it up
(newlines/extra spaces) — very useful for real pages.

### A complete extraction pattern
```python
titles = [t.text for t in soup.select("h2")]
prices = [p.text for p in soup.select(".price")]
links  = [a.get("href") for a in soup.select("a[href]")]
```
Collect **parallel lists** from **repeated containers**, then combine them.



In [ ]:
from bs4 import BeautifulSoup

html = '''<div class="card">
  <h2>Laptop</h2>
  <p class="price">₹55000</p>
  <a class="link" href="/laptop">view</a>
</div>
<div class="card">
  <h2>Tablet</h2>
  <p class="price">₹25000</p>
  <a class="link" href="/tablet">view</a>
</div>'''

soup = BeautifulSoup(html, "html.parser")

names  = [h.text             for h in soup.select("h2")]
prices = [p.text             for p in soup.select(".price")]
links  = [a.get("href")      for a in soup.select("a.link")]

print("names :", names)
print("prices:", prices)
print("links :", links)


## 8 · From HTML to a DataFrame (mini-project)

Now we combine everything: parse a page, extract repeated fields, and build a
pandas `DataFrame`.

### The workflow
1. Get the HTML (from `requests` or an in-memory string)
2. `BeautifulSoup` parse it
3. `soup.select(...)` the repeated **container** elements
4. Loop each container and **pull out the fields** (safer than parallel lists
   when some fields might be missing)
5. Assemble a list of dicts → `pd.DataFrame(list_of_dicts)`

> 👍 **Prefer per-container dicts** over parallel lists: if one product lacks a
> price, the rows still line up correctly.



In [ ]:
import pandas as pd
from bs4 import BeautifulSoup

page = '''<div class="card">
  <h2 class="name">Phone X</h2>
  <p class="price">₹15000</p>
  <span class="rating">4.5</span>
</div>
<div class="card">
  <h2 class="name">Phone Y</h2>
  <p class="price">₹20000</p>
  <span class="rating">4.2</span>
</div>
<div class="card">
  <h2 class="name">Phone Z</h2>
  <p class="price">₹25000</p>
  <span class="rating">4.8</span>
</div>'''

soup = BeautifulSoup(page, "html.parser")

rows = []
for card in soup.select("div.card"):
    rows.append({
        "name":   card.select_one(".name").text,
        "price":  card.select_one(".price").text,
        "rating": card.select_one(".rating").text,
    })

df_products = pd.DataFrame(rows)
df_products


## 9 · Pagination — scraping multiple pages

Real sites split results across many pages (`?page=1`, `?page=2`, ...). To scrape the
whole set you **loop over pages** and **accumulate** the rows.

### Pattern
```python
all_rows = []
for page_no in range(1, total_pages + 1):
    r = requests.get(url, params={"page": page_no}, headers=headers, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")
    all_rows.extend(extract(soup))     # combine with previous pages
    time.sleep(1)                      # be polite between requests
```
- **Respect `time.sleep(...)`** — hammering a server will get you blocked / banned.
- Watch the site's own "how many pages" (often in a `pagination` element).

Below is an offline simulation of the same idea using generated pages in memory.



In [ ]:
import time
from bs4 import BeautifulSoup

# Simulate 3 pages of a catalog (each like a real scraped page)
def make_page(items):
    cards = "".join(
        f'<div class="card"><h2 class="name">{n}</h2><p class="price">{p}</p></div>'
        for n, p in items
    )
    return f"<html><body>{cards}</body></html>"

pages = {
    1: make_page([("Phone A", "100"), ("Phone B", "200")]),
    2: make_page([("Phone C", "300"), ("Phone D", "400")]),
    3: make_page([("Phone E", "500")]),
}

all_rows = []
for page_no in range(1, len(pages) + 1):
    soup = BeautifulSoup(pages[page_no], "html.parser")
    for card in soup.select("div.card"):
        all_rows.append({
            "page": page_no,
            "name": card.select_one(".name").text,
            "price": card.select_one(".price").text,
        })
    time.sleep(0.1)     # in real code: sleep 1-2s to be polite

import pandas as pd
df_catalog = pd.DataFrame(all_rows)
print("Total items gathered across pages:", len(df_catalog))
df_catalog


## 10 · Robust Scraping (production-ready)

A scraper that works once often breaks the next day. Build in these safety nets:

### 1. Always use `try / except`
One bad page should **not** kill the whole job — skip it and continue.
```python
try:
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
except requests.exceptions.RequestException as e:
    print("Failed:", url, "->", e)
```

### 2. Handle status codes explicitly
- `429` / `403` → slow down, retry with backoff.
- Retry logic: try, wait, retry a few times, then give up.

### 3. Rate limiting
Add `time.sleep()` between requests. Be gentle.

### 4. Validate what you extracted
Check the row count / columns before saving — catch broken parsing early.



In [ ]:
import time
import requests
from bs4 import BeautifulSoup

def scrape_with_retry(url, headers, retries=3, delay=2):
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=10)
            if r.status_code == 200:
                return BeautifulSoup(r.text, "html.parser")
            elif r.status_code in (429, 503):      # rate limited / busy
                time.sleep(delay * (attempt + 1))  # exponential backoff
            else:
                print("Skipping", url, "status", r.status_code)
                return None
        except requests.exceptions.RequestException as e:
            print("Attempt", attempt + 1, "failed:", e)
            time.sleep(delay)
    return None

# Used offline below: scrape_with_retry on an in-memory page
soup = scrape_with_retry.__wrapped__ if hasattr(scrape_with_retry, "__wrapped__") else None
print("Robust scraper defined. In real use call it with a live URL.")


In [ ]:
from bs4 import BeautifulSoup

# Validating a scrape: always sanity-check the results
page = '''<div class="card"><h2 class="name">A</h2><p class="price">10</p></div>
<div class="card"><h2 class="name">B</h2><p class="price">20</p></div>'''
soup = BeautifulSoup(page, "html.parser")

rows = []
for card in soup.select("div.card"):
    name = card.select_one(".name")
    price = card.select_one(".price")
    if name and price:                      # skip incomplete cards
        rows.append({"name": name.text, "price": price.text})

print("extracted", len(rows), "rows")
expected = 2
assert len(rows) == expected, "Parsing changed — expected %d rows" % expected
print("Validation passed: row count matches expectation.")


## 11 · Selenium — scraping JavaScript-rendered pages

### The problem
Many modern sites build their HTML with **JavaScript** *after* the page loads.
`requests` only gets the empty shell — the data doesn't exist in the raw HTML.
`Selenium` launches a **real browser** that runs the JS, so you can read the
*rendered* page.

### Install
```bash
pip install selenium
```
Selenium Manager (bundled) auto-downloads the matching browser driver (ChromeDriver /
GeckoDriver) — you no longer need to download one manually.

### Basic workflow
```python
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

opts = Options()
opts.add_argument("--headless")               # invisible browser
driver = webdriver.Chrome(options=opts)
driver.get("https://example.com")

wait = WebDriverWait(driver, 10)
cards = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "card")))

for card in cards:
    print(card.find_element(By.TAG_NAME, "h2").text)

driver.quit()
```

### `By` locators
| `By` | find by | example |
|------|---------|---------|
| `By.ID` | id | `driver.find_element(By.ID, "email")` |
| `By.NAME` | name attr | `driver.find_element(By.NAME, "q")` |
| `By.CLASS_NAME` | class | `driver.find_element(By.CLASS_NAME, "price")` |
| `By.CSS_SELECTOR` | css | `driver.find_element(By.CSS_SELECTOR, "div.card h2")` |
| `By.XPATH` | xpath | `driver.find_element(By.XPATH, "//div[@class='card']")` |

### Waiting — explicit waits, never `time.sleep`
| Expected condition | waits until |
|--------------------|-------------|
| `presence_of_element_located` | element exists in DOM |
| `visibility_of_element_located` | element is visible |
| `element_to_be_clickable` | can be clicked |
| `presence_of_all_elements_located` | a list matches |

> ⚠️ `time.sleep()` is fragile (too short = flaky; too long = slow). Use **explicit waits**.



In [ ]:
# --- Selenium setup (ILLUSTRATION) ---
# Selenium is not installed in this notebook's environment, and launching a real
# browser here would be heavy — so the code is fully commented.
# Uncomment and run on a machine with `pip install selenium` + a browser to try it.
#
# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
#
# opts = Options()
# opts.add_argument("--headless")
# driver = webdriver.Chrome(options=opts)
# driver.get("https://quotes.toscrape.com/js/")
#
# wait = WebDriverWait(driver, 10)
# quotes = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "quote")))
#
# for q in quotes:
#     text = q.find_element(By.CLASS_NAME, "text").text
#     author = q.find_element(By.CLASS_NAME, "author").text
#     print(author, "->", text)
#
# driver.quit()
print("Selenium demo is shown as commented code (requires install + a browser).")


## 12 · Ethics & Legality

Scraping is powerful — use it responsibly.

### Check before you scrape
- **Terms of Service** — does the site forbid automated access?
- **`robots.txt`** — `https://site.com/robots.txt` lists disallowed paths.
- **`/robots.txt`** is the site's instructions to crawlers ("user-agent: * /no-scrape").

### Golden rules
1. **Prefer an official API** when one exists.
2. **Be slow & polite** — respect rate limits, add delays.
3. **Identify yourself** — set a real `User-Agent` / contact info.
4. **Don't bypass** logins, paywalls, CAPTCHAs, or access controls.
5. **Don't store personal data** you have no right to hold (privacy law).
6. **Respect copyright** — don't republish content without permission.
7. **Cache your results** — scrape once, don't re-fetch.

> ⚠️ This is general guidance, not legal advice. Laws differ by country
> (GDPR in Europe, etc.). When in doubt, get permission.



In [ ]:
# A minimal robots.txt parser pattern (for when you scrape at scale)
def check_robots(url):
    return ("Always read", url + "/robots.txt",
            "before writing an automated crawler")

print("robots.txt is the site's crawler rulebook:")
print(" -", "https://books.toscrape.com/robots.txt")
print(" -", "https://quotes.toscrape.com/robots.txt")


## 13 · Glossary & Cheatsheet

### Key terms
| Term | Meaning |
|------|---------|
| **HTTP** | protocol for fetching web resources |
| **GET / POST** | request methods (fetch / send) |
| **Status code** | 3-digit response result (200, 404...) |
| **User-Agent** | string identifying your client |
| **Header** | metadata sent with a request/response |
| **Session** | persistent connection + cookies |
| **DOM** | the parsed tree of a document |
| **Element** | one tag + content + attrs |
| **Attribute** | `name="value"` inside a tag |
| **CSS selector** | pattern to match elements |
| **Parser** | software that reads HTML into a tree |
| **robots.txt** | site's crawl instructions |
| **Rate limiting** | controlling request frequency |

### requests cheatsheet
```python
requests.get(url, params=..., headers=..., timeout=...)
requests.post(url, data=..., json=...)
session = requests.Session()
r.raise_for_status()
r.text / r.status_code / r.headers / r.url
```

### BeautifulSoup cheatsheet
```python
BeautifulSoup(html, "html.parser")
soup.find("tag")                    # first
soup.find_all("tag")                # all (list)
soup.select("css")                  # all by css
soup.select_one("css")              # first by css
el.text / el.get_text(" ", strip=True) / el["attr"] / el.get("attr")
el.find(...) / el.select(...)       # search within an element
```



In [ ]:
# Final combined cheatsheet as runnable reminders
from bs4 import BeautifulSoup
import pandas as pd

# 1) parse
soup = BeautifulSoup("<p class='x'>hi</p>", "html.parser")
# 2) find
el = soup.select_one("p.x")          # -> <p class='x'>hi</p>
# 3) extract
print("text:", el.text, "| class:", el.get("class"))

# 4) rows -> DataFrame (the universal final step)
data = [{"item": "A", "price": 10}, {"item": "B", "price": 20}]
df = pd.DataFrame(data)
df


## 14 · Practice Problems

Try these (in the notebook or on your own):

1. **parse** the HTML below, then
   - count the number of `<article>` elements,
   - print the text of every `<h2>`.

2. **extract** each article into a dict `{"title", "author", "summary"}` and build a
   `DataFrame`.

3. **filter** — print only articles whose author is "Alice".

4. **links** — collect every `href` on the page.

5. **(online)** scrape `https://quotes.toscrape.com/` — the quote text + author —
   into a DataFrame, respecting robots.txt and adding a delay.

6. **(online, harder)** `https://books.toscrape.com/` — scrape all book titles and
   prices, handling **pagination** (pages 1..50).



In [ ]:
from bs4 import BeautifulSoup
import pandas as pd

blog = '''<html><body>
  <article>
    <h2>First Post</h2>
    <p class="author">Alice</p>
    <p class="summary">About data science.</p>
  </article>
  <article>
    <h2>Second Post</h2>
    <p class="author">Bob</p>
    <p class="summary">About web scraping.</p>
  </article>
  <article>
    <h2>Third Post</h2>
    <p class="author">Alice</p>
    <p class="summary">About pandas.</p>
  </article>
</body></html>'''

soup = BeautifulSoup(blog, "html.parser")

# --- Exercise directions (fill in) ---
# 1) Count articles:
n_articles = len(soup.select("article"))
print("Number of articles:", n_articles)

# 2) Build the DataFrame
rows = []
for art in soup.select("article"):
    rows.append({
        "title":   art.select_one("h2").text,
        "author":  art.select_one(".author").text,
        "summary": art.select_one(".summary").text,
    })
df_blog = pd.DataFrame(rows)
print("\nAll posts:")
print(df_blog)

# 3) Filter by Alice:
print("\nOnly Alice's posts:")
print(df_blog[df_blog["author"] == "Alice"])


## Wrap-Up

You now know the **full scraping pipeline**:
```
requests (download HTML)
   ↓
BeautifulSoup (parse into a tree)
   ↓
soup.select(...) (find the repeated elements)
   ↓
extract text / attributes
   ↓
pandas DataFrame (clean, structured data)
   ↓
save / analyze
```
And you know when to level up to **Selenium** for JavaScript-rendered sites,
plus how to be **fast, reliable, and ethical**.

> **Next steps:** try `quotes.toscrape.com` and `books.toscrape.com` for risk-free
> practice (they exist explicitly for learning scraping), then a real site you have
> permission to use.

